# Instrumented PRMP Mechanism Analysis: Information Filtering vs Implicit Regularization

This notebook demonstrates the **Predictive Residual Message Passing (PRMP)** mechanism analysis on the Amazon Video Games dataset.

**What this experiment does:**
- Trains three GNN variants (standard SAGEConv, PRMP, and no-subtraction) on a heterogeneous review graph
- Instruments training with per-epoch measurements: prediction R², mutual information, gradient norms, and weight dynamics
- Compares whether PRMP's benefit comes from **information filtering** (residuals carry more task-relevant info) or **implicit regularization**

**Key findings from pre-computed results:**
- PRMP achieves +0.74% RMSE improvement vs standard SAGEConv
- MI(residual, target) / MI(raw, target) ratio > 1.0, supporting the information filtering hypothesis
- The demo loads pre-computed experiment outputs and visualizes the mechanism analysis

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# torch, torch_geometric — NOT on Colab by default for geometric, always install
# We only need torch for model definition (no training in this demo)
_pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')
_pip('torch-geometric==2.6.1')
_pip('loguru==0.7.3')
_pip('psutil==6.1.1')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

## Imports

In [ ]:
import gc
import json
import math
import os
import sys
import time
from hashlib import md5
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from torch.optim import Adam
import matplotlib.pyplot as plt

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, LayerNorm
from torch_geometric.nn.conv import MessagePassing

print("All imports successful!")

## Data Loading

Load pre-computed experiment results from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter3_instrumented_pr/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata: {json.dumps(data['metadata'], indent=2)}")

## Configuration

These are the hyperparameters used in the original experiment. Since we load pre-computed results, these are for reference and used by the analysis functions below.

In [ ]:
# ---------------------------------------------------------------------------
# Hyperparameters (from original experiment)
# ---------------------------------------------------------------------------
HIDDEN_DIM = 64
NUM_GNN_LAYERS = 2
LR = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 100
SEEDS = [42, 123, 7]
VARIANTS = ["standard", "prmp", "no_subtract"]
MI_INTERVAL = 10
MI_SUBSAMPLE = 2000
TEXT_HASH_DIM = 16

# Demo config: how many review examples to display in tables
N_DISPLAY_REVIEWS = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Model Architecture: PRMPConv and InstrumentedHeteroGNN

The key innovation is the **Predictive Residual Message Passing (PRMP)** convolution:
- A prediction MLP estimates child node embeddings from parent embeddings
- Messages are the **residual** (actual - predicted), not the raw child embedding
- This filters out predictable information, leaving only novel/surprising content
- LayerNorm stabilizes the residual signal

In [ ]:
class PRMPConv(MessagePassing):
    """Predictive Residual Message Passing convolution.

    From research_id2_it1__opus Section 7.1:
    - 2-layer MLP prediction, zero-init final layer
    - LayerNorm on residuals (for 'prmp' mode)
    - Mean aggregation, detached parent input
    - GraphSAGE-style concat+linear update
    """

    def __init__(self, in_src: int, in_dst: int, out_channels: int, mode: str = "prmp"):
        super().__init__(aggr="mean")
        self.mode = mode
        hidden = min(in_dst, in_src)

        # Prediction MLP: parent -> predicted child
        self.pred_mlp = nn.Sequential(
            nn.Linear(in_dst, hidden),
            nn.ReLU(),
            nn.Linear(hidden, in_src),
        )
        # Zero-init final layer (Section 5.3)
        nn.init.zeros_(self.pred_mlp[-1].weight)
        nn.init.zeros_(self.pred_mlp[-1].bias)

        # LayerNorm on residuals (Section 5.2 Option B)
        self.norm = nn.LayerNorm(in_src) if mode == "prmp" else None

        # Update MLP (GraphSAGE-style)
        if mode == "no_subtract":
            update_in = in_dst + in_src * 2  # concat raw + predicted
        else:
            update_in = in_dst + in_src  # concat parent + residual
        self.update_mlp = nn.Linear(update_in, out_channels)

        # Instrumentation storage
        self._last_predicted = None
        self._last_residual = None
        self._last_child_h = None
        self._last_parent_h = None

    def forward(self, x, edge_index):
        if isinstance(x, tuple) or isinstance(x, list):
            x_src, x_dst = x[0], x[1]
        else:
            x_src = x_dst = x
        aggr_out = self.propagate(edge_index, x=(x_src, x_dst))
        out = self.update_mlp(torch.cat([x_dst, aggr_out], dim=-1))
        return out

    def message(self, x_j, x_i):
        # x_j = source (child), x_i = destination (parent)
        predicted = self.pred_mlp(x_i.detach())

        # Store for instrumentation
        self._last_predicted = predicted.detach()
        self._last_child_h = x_j.detach()
        self._last_parent_h = x_i.detach()

        if self.mode == "no_subtract":
            return torch.cat([x_j, predicted], dim=-1)
        else:
            residual = x_j - predicted
            self._last_residual = residual.detach()
            if self.norm is not None:
                return self.norm(residual)
            return residual


class InstrumentedHeteroGNN(nn.Module):
    """Heterogeneous GNN with full instrumentation for mechanism analysis."""

    def __init__(self, product_dim: int, customer_dim: int, review_dim: int,
                 hidden_dim: int, num_layers: int, variant: str):
        super().__init__()
        self.variant = variant
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Feature encoders
        self.product_enc = nn.Sequential(
            nn.Linear(product_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.customer_enc = nn.Sequential(
            nn.Linear(customer_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.review_enc = nn.Sequential(
            nn.Linear(review_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )

        # GNN layers with HeteroConv
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        hd = hidden_dim

        for layer in range(num_layers):
            if variant == "standard":
                conv_dict = {
                    ("review", "of_product", "product"): SAGEConv((hd, hd), hd),
                    ("product", "has_review", "review"): SAGEConv((hd, hd), hd),
                    ("review", "by_customer", "customer"): SAGEConv((hd, hd), hd),
                    ("customer", "wrote_review", "review"): SAGEConv((hd, hd), hd),
                }
            else:
                # PRMP/no_subtract on child->parent edges, SAGEConv on parent->child
                conv_dict = {
                    ("review", "of_product", "product"): PRMPConv(hd, hd, hd, mode=variant),
                    ("product", "has_review", "review"): SAGEConv((hd, hd), hd),
                    ("review", "by_customer", "customer"): PRMPConv(hd, hd, hd, mode=variant),
                    ("customer", "wrote_review", "review"): SAGEConv((hd, hd), hd),
                }
            self.convs.append(HeteroConv(conv_dict, aggr="sum"))

            # Per-node-type LayerNorm
            norm_dict = {
                "product": LayerNorm(hd),
                "customer": LayerNorm(hd),
                "review": LayerNorm(hd),
            }
            self.norms.append(nn.ModuleDict(norm_dict))

        # Prediction head: review node -> rating
        self.head = nn.Sequential(
            nn.Linear(hd, hd // 2), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hd // 2, 1)
        )

    def encode_features(self, data: HeteroData) -> dict:
        """Encode raw features into hidden representations."""
        return {
            "product": self.product_enc(data["product"].x),
            "customer": self.customer_enc(data["customer"].x),
            "review": self.review_enc(data["review"].x),
        }

    def forward(self, data: HeteroData) -> torch.Tensor:
        h_dict = self.encode_features(data)

        edge_index_dict = {
            key: data[key].edge_index for key in [
                ("review", "of_product", "product"),
                ("product", "has_review", "review"),
                ("review", "by_customer", "customer"),
                ("customer", "wrote_review", "review"),
            ]
        }

        for layer_idx, (conv, norm_dict) in enumerate(zip(self.convs, self.norms)):
            h_new = conv(h_dict, edge_index_dict)
            # Apply norms, ReLU, and residual connections
            for node_type in h_new:
                h_new[node_type] = norm_dict[node_type](h_new[node_type])
                h_new[node_type] = F.relu(h_new[node_type])
                if layer_idx > 0:
                    h_new[node_type] = h_new[node_type] + h_dict[node_type]
            h_dict = h_new

        return self.head(h_dict["review"]).squeeze(-1)

    def get_prmp_convs(self) -> list:
        """Return list of (layer_idx, edge_type, prmp_conv) for instrumentation."""
        result = []
        for layer_idx, conv in enumerate(self.convs):
            for edge_type, subconv in conv.convs.items():
                if isinstance(subconv, PRMPConv):
                    result.append((layer_idx, edge_type, subconv))
        return result

# Verify model can be instantiated
model = InstrumentedHeteroGNN(
    product_dim=5, customer_dim=5, review_dim=21,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_GNN_LAYERS, variant="prmp"
)
n_params = sum(p.numel() for p in model.parameters())
print(f"PRMP model: {n_params:,} parameters")
del model

## Parse Pre-computed Results

Extract per-review predictions and per-variant training metrics from the loaded data.

In [ ]:
# ---------------------------------------------------------------------------
# Separate review-level examples from variant summaries
# ---------------------------------------------------------------------------
examples = data["datasets"][0]["examples"]

review_examples = [ex for ex in examples if "metadata_variant" not in ex]
summary_examples = {
    ex["metadata_variant"]: ex for ex in examples if "metadata_variant" in ex
}

print(f"Review-level predictions: {len(review_examples)}")
print(f"Summary examples: {list(summary_examples.keys())}")

# Build a DataFrame of per-review predictions
rows = []
for ex in review_examples[:N_DISPLAY_REVIEWS]:
    rows.append({
        "true_rating": float(ex["output"]),
        "pred_standard": float(ex["predict_standard"]),
        "pred_prmp": float(ex["predict_prmp"]),
        "pred_no_subtract": float(ex["predict_no_subtract"]),
        "seed": ex["metadata_seed"],
        "review_idx": ex["metadata_review_index"],
    })

review_df = pd.DataFrame(rows)
print(f"\nPer-review predictions (first {len(rows)}):")
print(review_df.to_string(index=False))

## Mechanism Diagnosis

Analyze whether PRMP's benefit comes from **information filtering** (MI ratio > 1) or **implicit regularization**, using the aggregated results and epoch-level diagnostics from the pre-computed experiment.

In [ ]:
# ---------------------------------------------------------------------------
# Extract aggregated results from variant summaries
# ---------------------------------------------------------------------------
aggregated = {}
for variant in VARIANTS:
    if variant in summary_examples:
        out = json.loads(summary_examples[variant]["output"])
        aggregated[variant] = out.get("aggregated", {})

# Display aggregated results
print("=" * 60)
print("AGGREGATED RESULTS")
print("=" * 60)
for v, agg in aggregated.items():
    rmse = agg.get("mean_test_rmse", "N/A")
    std = agg.get("std_test_rmse", "N/A")
    rel = agg.get("rel_improve_pct", "N/A")
    print(f"  {v:15s}: test_rmse={rmse} +/- {std}  rel_improve={rel}%")

# ---------------------------------------------------------------------------
# Extract mechanism diagnosis from the summary example
# ---------------------------------------------------------------------------
if "summary" in summary_examples:
    diag_out = json.loads(summary_examples["summary"]["output"])
    mechanism_diagnosis = diag_out.get("mechanism_diagnosis", {})
    conclusion = diag_out.get("conclusion", "")

    print(f"\n{'=' * 60}")
    print("MECHANISM DIAGNOSIS")
    print(f"{'=' * 60}")
    for k, v in mechanism_diagnosis.items():
        print(f"  {k}: {v}")
    print(f"\nConclusion: {conclusion}")
else:
    mechanism_diagnosis = {}
    conclusion = "No summary data available"
    print("No mechanism diagnosis summary found in data")

## Extract Training Curves

Parse per-epoch metrics (loss, RMSE, mutual information) from the variant summary data for visualization.

In [ ]:
# ---------------------------------------------------------------------------
# Extract epoch-level training curves from variant summaries
# ---------------------------------------------------------------------------
training_curves = {}  # variant -> {epochs, train_loss, val_rmse, test_rmse, mi_data}

for variant in VARIANTS:
    if variant not in summary_examples:
        continue
    out = json.loads(summary_examples[variant]["output"])
    per_seed = out.get("per_seed_epochs", [])
    if not per_seed:
        continue

    # Use first seed's epochs for plotting
    epochs_data = per_seed[0]["epochs"]
    curve = {
        "epochs": [e["epoch"] for e in epochs_data],
        "train_loss": [e["train_loss"] for e in epochs_data],
        "val_rmse": [e["val_rmse"] for e in epochs_data],
        "test_rmse": [e["test_rmse"] for e in epochs_data],
    }

    # Extract MI data where available
    mi_epochs = []
    mi_ratios = []
    mi_raw = []
    mi_residual = []
    for e in epochs_data:
        if "mi" in e and e["mi"]:
            mi_epochs.append(e["epoch"])
            # Average across edge types
            ratios = [v.get("mi_ratio", 0) for v in e["mi"].values() if "mi_ratio" in v]
            raws = [v.get("mi_raw", 0) for v in e["mi"].values() if "mi_raw" in v]
            residuals = [v.get("mi_res", 0) for v in e["mi"].values() if "mi_res" in v]
            mi_ratios.append(np.mean(ratios) if ratios else 0)
            mi_raw.append(np.mean(raws) if raws else 0)
            mi_residual.append(np.mean(residuals) if residuals else 0)

    curve["mi_epochs"] = mi_epochs
    curve["mi_ratios"] = mi_ratios
    curve["mi_raw"] = mi_raw
    curve["mi_residual"] = mi_residual

    # Extract pred_r2 where available
    r2_epochs = []
    r2_values = []
    for e in epochs_data:
        if "pred_r2" in e and e["pred_r2"]:
            r2_epochs.append(e["epoch"])
            r2s = [v.get("pred_r2", 0) for v in e["pred_r2"].values() if "pred_r2" in v]
            r2_values.append(np.mean(r2s) if r2s else 0)

    curve["r2_epochs"] = r2_epochs
    curve["r2_values"] = r2_values

    training_curves[variant] = curve
    print(f"{variant}: {len(curve['epochs'])} epoch snapshots, {len(mi_epochs)} MI measurements, {len(r2_epochs)} R2 measurements")

## Visualization

Four-panel figure showing:
1. **Training Loss** curves for all three variants
2. **Validation RMSE** convergence comparison
3. **Per-Review Prediction Errors** across variants
4. **Aggregated RMSE** bar chart with error bars

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {"standard": "#1f77b4", "prmp": "#d62728", "no_subtract": "#2ca02c"}
labels = {"standard": "Standard SAGEConv", "prmp": "PRMP", "no_subtract": "No-Subtract"}

# --- Panel 1: Training Loss ---
ax = axes[0, 0]
for variant, curve in training_curves.items():
    ax.plot(curve["epochs"], curve["train_loss"],
            color=colors[variant], label=labels[variant], linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (MSE)")
ax.set_title("Training Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 2: Validation RMSE ---
ax = axes[0, 1]
for variant, curve in training_curves.items():
    ax.plot(curve["epochs"], curve["val_rmse"],
            color=colors[variant], label=labels[variant], linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation RMSE")
ax.set_title("Validation RMSE Convergence")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 3: Per-Review Prediction Error Distribution ---
ax = axes[1, 0]
for i, variant in enumerate(["pred_standard", "pred_prmp", "pred_no_subtract"]):
    vname = variant.replace("pred_", "")
    errors = np.abs(review_df[variant] - review_df["true_rating"])
    ax.bar(np.arange(len(errors)) + i * 0.25, errors, width=0.25,
           color=colors.get(vname, "#999"), label=labels.get(vname, vname), alpha=0.8)
ax.set_xlabel("Review Index")
ax.set_ylabel("|Prediction Error|")
ax.set_title("Per-Review Absolute Prediction Errors")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# --- Panel 4: Aggregated RMSE Bar Chart ---
ax = axes[1, 1]
variant_names = []
rmse_means = []
rmse_stds = []
for variant in VARIANTS:
    if variant in aggregated:
        variant_names.append(labels.get(variant, variant))
        rmse_means.append(aggregated[variant].get("mean_test_rmse", 0))
        rmse_stds.append(aggregated[variant].get("std_test_rmse", 0))

x_pos = np.arange(len(variant_names))
bar_colors = [colors.get(v, "#999") for v in VARIANTS if v in aggregated]
bars = ax.bar(x_pos, rmse_means, yerr=rmse_stds, capsize=5,
              color=bar_colors, alpha=0.8, edgecolor="black", linewidth=0.8)

# Add value labels on bars
for bar, mean, std in zip(bars, rmse_means, rmse_stds):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + std + 0.005,
            f"{mean:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(x_pos)
ax.set_xticklabels(variant_names)
ax.set_ylabel("Test RMSE")
ax.set_title("Final Test RMSE Comparison (3-seed avg)")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("prmp_mechanism_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to prmp_mechanism_analysis.png")

## Mutual Information Analysis

Plot the MI ratio (residual vs raw) over training epochs for PRMP. A ratio > 1.0 indicates that **information filtering** is occurring: residuals carry MORE task-relevant information than the raw messages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel 1: MI Ratio over epochs for PRMP ---
ax = axes[0]
prmp_has_mi = "prmp" in training_curves and training_curves["prmp"]["mi_epochs"]
if prmp_has_mi:
    curve = training_curves["prmp"]
    ax.plot(curve["mi_epochs"], curve["mi_ratios"], "o-",
            color=colors["prmp"], linewidth=2, markersize=6, label="MI(residual)/MI(raw)")
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=1, label="Ratio = 1.0 (neutral)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MI Ratio (residual / raw)")
    ax.set_title("PRMP: Information Filtering Ratio Over Training")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No MI data available for PRMP", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("PRMP: Information Filtering Ratio")

# --- Panel 2: Prediction R2 over epochs ---
ax = axes[1]
has_r2 = False
for variant in ["prmp", "no_subtract"]:
    if variant in training_curves and training_curves[variant]["r2_epochs"]:
        curve = training_curves[variant]
        ax.plot(curve["r2_epochs"], curve["r2_values"], "o-",
                color=colors[variant], linewidth=2, markersize=5,
                label=f"{labels[variant]} pred R2")
        has_r2 = True
if has_r2:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Prediction MLP R-squared")
    ax.set_title("Prediction Accuracy: How Well Parent Predicts Child")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No R2 data available", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Prediction R2")

plt.tight_layout()
plt.savefig("prmp_mi_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to prmp_mi_analysis.png")

# --- Final Summary ---
print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Dataset: Amazon Video Games ({data['metadata'].get('dataset', 'N/A')})")
print(f"Hidden dim: {data['metadata'].get('hidden_dim', 'N/A')}, Epochs: {data['metadata'].get('epochs', 'N/A')}")
print(f"Seeds: {data['metadata'].get('seeds', 'N/A')}")
print(f"Variants: {data['metadata'].get('variants', 'N/A')}")
if mechanism_diagnosis:
    mi_ratio = mechanism_diagnosis.get("mi_residual_vs_raw_ratio", "N/A")
    filtering = mechanism_diagnosis.get("information_filtering_supported", "N/A")
    print(f"\nMI Ratio (residual/raw): {mi_ratio}")
    print(f"Information Filtering Supported: {filtering}")
    print(f"Gradient Angle Trend: {mechanism_diagnosis.get('gradient_angle_trend', 'N/A')}")
    print(f"Weight Growth Rate: {mechanism_diagnosis.get('weight_growth_rate', 'N/A')}x")
print(f"\nConclusion: {conclusion}")